# Deployment Validation Check

**Purpose:** Verify all required files and paths before running pipeline.

**Checks:** Machine config, data files, config CSVs, paths, key columns

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import yaml, os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'lib'))
from utils import setup_notebook_environment, get_machine_id, load_config

print("="*60)
print("  DEPLOYMENT VALIDATION CHECK")
print("="*60)

project_root = setup_notebook_environment()
os.chdir(project_root)

issues, checks_passed, checks_total = [], 0, 0

def check(name, condition, error_msg=None):
    global checks_passed, checks_total, issues
    checks_total += 1
    if condition:
        print(f"✓ {name}")
        checks_passed += 1
        return True
    else:
        msg = error_msg if error_msg else name
        print(f"✗ {name}: {msg}")
        issues.append(f"{name}: {msg}")
        return False

## 1. Machine Configuration

In [ ]:
print("\n[1] Machine Configuration\n" + "-"*40)
pc_file = project_root / "current.pc"
check("current.pc exists", pc_file.exists())
machine_id = get_machine_id() if pc_file.exists() else None
if machine_id:
    check("current.pc valid", machine_id in [1,2,3,4])
    print(f"  → PC{machine_id}")
config_file = project_root / config_path / "config.yaml"
check("config.yaml exists", config_file.exists())

cfg = None
if config_file.exists() and machine_id:
    try:
        cfg = load_config(str(config_file))
        check("config.yaml loads", True)
        mk = f"PC{machine_id}"
        check(f"{mk} section exists", mk in cfg.get('machines', {}))
        if mk in cfg.get('machines', {}):
            paths = cfg['machines'][mk]['paths']
            check("master_data_path set", paths.get('master_data_path') and paths['master_data_path'] != 'null')
            check("aux_data_path set", paths.get('aux_data_path') and paths['aux_data_path'] != 'null')
            print(f"\n  Paths:\n    master: {paths.get('master_data_path')}\n    aux:    {paths.get('aux_data_path')}")
    except Exception as e:
        check("config.yaml loads", False, str(e))

## 2. Data Files

In [ ]:
print("\n[2] Data Files\n" + "-"*40)
files_info = {}
if cfg and machine_id:
    paths = cfg['machines'][f'PC{machine_id}']['paths']
    data_cfg = cfg['data']
    mp = Path(paths['master_data_path']) if paths.get('master_data_path') else None
    ap = Path(paths['aux_data_path']) if paths.get('aux_data_path') else None
    
    if mp:
        check("master_data_path exists", mp.exists())
        mf = mp / data_cfg['master_file']
        exists = mf.exists()
        check(f"master_file", exists)
        files_info['master'] = {'path': mf, 'exists': exists}
    
    if ap:
        check("aux_data_path exists", ap.exists())
        af = ap / data_cfg['aux_file']
        exists = af.exists()
        check(f"aux_file", exists)
        files_info['aux'] = {'path': af, 'exists': exists}
        
        cf = ap / data_cfg['control_model_file']
        exists = cf.exists()
        check(f"GLM control_file", exists)
        files_info['glm'] = {'path': cf, 'exists': exists}
    
    print("\n  Sizes:")
    for n, i in files_info.items():
        if i['exists']:
            print(f"    {n:10s}: {i['path'].stat().st_size/(1024*1024):>8.1f} MB")
else:
    print("⚠ Skipped")

## 3. Config Files

In [ ]:
print("\n[3] Config Files\n" + "-"*40)
config_dir = project_root / config_path
for fname in ['monotonicity.csv', 'exclusion.csv', 'manual_feature_encoding.csv', 'columns_to_load_during_dataassembly.csv', 'columns_inclusion.csv', 'pca_features.csv']:
    check(fname, (config_dir / fname).exists())

## 4. Data Validation

In [ ]:
print("\n[4] Data Validation\n" + "-"*40)
if cfg and 'files_info' in locals() and all(f['exists'] for f in files_info.values()):
    jk = cfg['data']['join_key']
    fc = cfg['data']['fold_column']
    tgt = cfg['experiment']['target']
    exp = cfg['experiment']['exposure']
    cov = cfg['experiment']['coverage']
    
    for name, finfo in files_info.items():
        try:
            pqf = pq.ParquetFile(finfo['path'])
            cols = pqf.schema.names
            print(f"\n{name.upper()}: {pqf.metadata.num_rows:,} rows, {len(cols)} cols")
            check(f"  has '{jk}'", jk in cols)
            if name == 'aux':
                check(f"  has '{fc}'", fc in cols)
                check(f"  has '{tgt}'", tgt in cols)
                check(f"  has '{exp}'", exp in cols)
            if name == 'glm':
                glm_col = f"pred_pp_{cov}"
                check(f"  has '{glm_col}'", glm_col in cols)
        except Exception as e:
            check(f"{name} readable", False, str(e))
else:
    print("⚠ Skipped")

## Summary

In [ ]:
print("\n" + "="*60)
print("  SUMMARY")
print("="*60)
print(f"\nChecks: {checks_passed}/{checks_total}")
if issues:
    print(f"\n✗ Issues ({len(issues)}):")
    for i in issues: print(f"  - {i}")
    print("\n❌ NOT READY")
else:
    print("\n✅ READY TO RUN")
print("="*60)